In [ ]:
"""
Sistem Query Semantik - Domain Akademik
========================================
Relasi:
  mahasiswa  → mengambil  → matkul
  dosen      → mengajar   → matkul
"""

from collections import defaultdict


# ─────────────────────────────────────────
# 1. KNOWLEDGE GRAPH
# ─────────────────────────────────────────

class KnowledgeGraph:
    def __init__(self):
        self.triples: list = []
        self.index = defaultdict(list)

    def tambah(self, subjek, predikat, objek):
        triple = (subjek.lower(), predikat.lower(), objek.lower())
        if triple not in self.triples:
            self.triples.append(triple)
            self.index[(subjek.lower(), predikat.lower())].append(objek.lower())
            self.index[(None, predikat.lower(), objek.lower())].append(subjek.lower())
            return True
        return False

    def hapus(self, subjek, predikat, objek):
        triple = (subjek.lower(), predikat.lower(), objek.lower())
        if triple in self.triples:
            self.triples.remove(triple)
            self.index[(subjek.lower(), predikat.lower())].remove(objek.lower())
            self.index[(None, predikat.lower(), objek.lower())].remove(subjek.lower())
            return True
        return False

    def tanya_objek(self, subjek, predikat):
        return self.index.get((subjek.lower(), predikat.lower()), [])

    def tanya_subjek(self, predikat, objek):
        return self.index.get((None, predikat.lower(), objek.lower()), [])

    def query(self, subjek=None, predikat=None, objek=None):
        hasil = []
        for s, p, o in self.triples:
            if ((subjek   is None or s == subjek.lower()) and
                (predikat is None or p == predikat.lower()) and
                (objek    is None or o == objek.lower())):
                hasil.append((s, p, o))
        return hasil

    def tampilkan_semua(self):
        if not self.triples:
            print("  (belum ada data)")
            return
        mhs = [(s,p,o) for s,p,o in self.triples if p == "mengambil"]
        dsn = [(s,p,o) for s,p,o in self.triples if p == "mengajar"]
        if mhs:
            print("  [MAHASISWA -> mengambil -> MATKUL]")
            for s,p,o in mhs:
                print(f"    {s}  ->  {o}")
        if dsn:
            print("  [DOSEN -> mengajar -> MATKUL]")
            for s,p,o in dsn:
                print(f"    {s}  ->  {o}")


# ─────────────────────────────────────────
# 2. SEMANTIC QUERY PARSER
# ─────────────────────────────────────────

class SemanticQueryParser:
    PREDIKAT_MAP = {
        "mengambil":"mengambil","ambil":"mengambil","diambil":"mengambil",
        "mengajar":"mengajar","ajar":"mengajar","diajarkan":"mengajar","diajar":"mengajar",
    }

    def __init__(self, kg):
        self.kg = kg

    def parse_dan_eksekusi(self, teks):
        token = teks.lower().strip().rstrip("?").split()
        predikat_ditemukan, idx_predikat = None, -1
        for i, kata in enumerate(token):
            if kata in self.PREDIKAT_MAP:
                predikat_ditemukan = self.PREDIKAT_MAP[kata]
                idx_predikat = i
                break

        if predikat_ditemukan is None:
            return "Tidak mengenali predikat. Gunakan 'mengambil' atau 'mengajar'."

        sisa = token[idx_predikat + 1:]
        buang = {"yang","oleh","dari","ke","di","dengan","adalah","itu","ini","matkul","mata","kuliah"}
        sisa_bersih = [w for w in sisa if w not in buang]

        if not sisa_bersih:
            hasil = self.kg.query(predikat=predikat_ditemukan)
            if not hasil:
                return f"Tidak ada data untuk '{predikat_ditemukan}'."
            return "Semua relasi:\n" + "\n".join(f"  {s} -> {p} -> {o}" for s,p,o in hasil)

        argumen = " ".join(sisa_bersih)
        arah_objek = any(k in token for k in ["diambil","diajarkan","diajar"])

        if arah_objek:
            obj_list = self.kg.tanya_objek(argumen, predikat_ditemukan)
            if not obj_list:
                return f"Tidak ditemukan data untuk '{argumen}'."
            return f"'{argumen}' {predikat_ditemukan}:\n  -> " + "\n  -> ".join(obj_list)
        else:
            subj_list = self.kg.tanya_subjek(predikat_ditemukan, argumen)
            if not subj_list:
                return f"Tidak ada yang '{predikat_ditemukan}' '{argumen}'."
            label = "Mahasiswa" if predikat_ditemukan == "mengambil" else "Dosen"
            return f"{label} yang '{predikat_ditemukan}' '{argumen}':\n  -> " + "\n  -> ".join(subj_list)


# ─────────────────────────────────────────
# 3. MENU INPUT INTERAKTIF
# ─────────────────────────────────────────

def garis(char="-", n=55):
    print(char * n)

def menu_tambah_data(kg):
    garis()
    print("  TAMBAH DATA")
    garis()
    print("  [1] Mahasiswa mengambil matkul")
    print("  [2] Dosen mengajar matkul")
    print("  [0] Batal")
    pilihan = input("  > ").strip()

    if pilihan == "0":
        return
    elif pilihan == "1":
        nama   = input("  Nama mahasiswa : ").strip()
        matkul = input("  Nama matkul    : ").strip()
        if nama and matkul:
            ok = kg.tambah(nama, "mengambil", matkul)
            print("  [OK] Ditambahkan!" if ok else "  [!] Data sudah ada.")
        else:
            print("  [!] Nama/matkul tidak boleh kosong.")
    elif pilihan == "2":
        nama   = input("  Nama dosen  : ").strip()
        matkul = input("  Nama matkul : ").strip()
        if nama and matkul:
            ok = kg.tambah(nama, "mengajar", matkul)
            print("  [OK] Ditambahkan!" if ok else "  [!] Data sudah ada.")
        else:
            print("  [!] Nama/matkul tidak boleh kosong.")
    else:
        print("  [!] Pilihan tidak valid.")


def menu_hapus_data(kg):
    garis()
    print("  HAPUS DATA")
    garis()
    print("  [1] Mahasiswa mengambil matkul")
    print("  [2] Dosen mengajar matkul")
    print("  [0] Batal")
    pilihan = input("  > ").strip()

    if pilihan == "0":
        return
    if pilihan in ("1", "2"):
        predikat = "mengambil" if pilihan == "1" else "mengajar"
        label_s  = "mahasiswa" if pilihan == "1" else "dosen"
        nama   = input(f"  Nama {label_s} : ").strip()
        matkul = input("  Nama matkul  : ").strip()
        ok = kg.hapus(nama, predikat, matkul)
        print("  [OK] Data dihapus." if ok else "  [!] Data tidak ditemukan.")
    else:
        print("  [!] Pilihan tidak valid.")


def menu_input_massal(kg):
    garis()
    print("  INPUT MASSAL")
    garis()
    print("  Format : <nama> | <mengambil/mengajar> | <matkul>")
    print("  Contoh : Budi | mengambil | Basis Data")
    print("  Ketik 'selesai' jika sudah.\n")
    tambah = 0
    while True:
        baris = input("  > ").strip()
        if baris.lower() in ("selesai", "done", ""):
            break
        bagian = [b.strip() for b in baris.split("|")]
        if len(bagian) != 3:
            print("  [!] Format salah. Harus: nama | predikat | matkul")
            continue
        nama, predikat, matkul = bagian
        pred_map = {"mengambil":"mengambil","ambil":"mengambil",
                    "mengajar":"mengajar","ajar":"mengajar"}
        predikat_normal = pred_map.get(predikat.lower())
        if not predikat_normal:
            print("  [!] Predikat harus 'mengambil' atau 'mengajar'.")
            continue
        ok = kg.tambah(nama, predikat_normal, matkul)
        status = "[OK]" if ok else "[duplikat]"
        print(f"  {status} {nama} -> {predikat_normal} -> {matkul}")
        if ok:
            tambah += 1
    print(f"\n  Total {tambah} data berhasil ditambahkan.")


# ─────────────────────────────────────────
# 4. DATA AWAL
# ─────────────────────────────────────────

def buat_kg_awal():
    kg = KnowledgeGraph()
    kg.tambah("Tsara Azizah Effendhi",       "mengambil", "Basis Data")
    kg.tambah("Tsara Azizah Effendhi",       "mengambil", "Pemrograman Web")
    kg.tambah("Adittya Bayu Syahputra",      "mengambil", "Basis Data")
    kg.tambah("Adittya Bayu Syahputra",      "mengambil", "Kecerdasan Buatan")
    kg.tambah("Mukhammad Nasyaril Ferdianto","mengambil", "Pemrograman Web")
    kg.tambah("Moch Rifky Sabilul Khaqiq",   "mengambil", "Kecerdasan Buatan")
    kg.tambah("Muhammad Agus Indra Hermawan","mengambil", "Jaringan Komputer")
    kg.tambah("Bapak Anggasoft spoken",  "mengajar", "Basis Data")
    kg.tambah("Ibu Neny Kurniati",   "mengajar", "Pemrograman Web")
    kg.tambah("Bapak Anggay Lury", "mengajar", "Kecerdasan Buatan")
    kg.tambah("Bapak Machlul Alamin",  "mengajar", "Jaringan Komputer")
    return kg


# ─────────────────────────────────────────
# 5. MAIN LOOP
# ─────────────────────────────────────────

def main():
    kg     = buat_kg_awal()
    parser = SemanticQueryParser(kg)

    while True:
        print()
        garis("=")
        print("  SISTEM QUERY SEMANTIK - AKADEMIK")
        garis("=")
        print("  [1] Lihat semua data")
        print("  [2] Tambah data  (satu per satu)")
        print("  [3] Input massal (nama | predikat | matkul)")
        print("  [4] Hapus data")
        print("  [5] Query semantik")
        print("  [0] Keluar")
        garis()
        pilihan = input("  Pilih menu > ").strip()

        if pilihan == "0":
            print("  Sampai jumpa!")
            break
        elif pilihan == "1":
            garis()
            print("  DATA SAAT INI:")
            garis()
            kg.tampilkan_semua()
        elif pilihan == "2":
            menu_tambah_data(kg)
        elif pilihan == "3":
            menu_input_massal(kg)
        elif pilihan == "4":
            menu_hapus_data(kg)
        elif pilihan == "5":
            garis()
            print("  QUERY SEMANTIK")
            print("  Contoh: 'siapa yang mengambil basis data'")
            print("          'apa yang diajarkan Bapak anggasoft spoken'")
            print("  Ketik 'kembali' untuk ke menu utama.")
            garis()
            while True:
                q = input("\n  Query > ").strip()
                if q.lower() in ("kembali", "back", ""):
                    break
                print("  ->", parser.parse_dan_eksekusi(q))
        else:
            print("  [!] Pilihan tidak valid.")


if __name__ == "__main__":
    main()



  SISTEM QUERY SEMANTIK - AKADEMIK
  [1] Lihat semua data
  [2] Tambah data  (satu per satu)
  [3] Input massal (nama | predikat | matkul)
  [4] Hapus data
  [5] Query semantik
  [0] Keluar
-------------------------------------------------------
-------------------------------------------------------
  DATA SAAT INI:
-------------------------------------------------------
  [MAHASISWA -> mengambil -> MATKUL]
    tsara azizah effendhi  ->  basis data
    tsara azizah effendhi  ->  pemrograman web
    adittya bayu syahputra  ->  basis data
    adittya bayu syahputra  ->  kecerdasan buatan
    mukhammad nasyaril ferdianto  ->  pemrograman web
    moch rifky sabilul khaqiq  ->  kecerdasan buatan
    muhammad agus indra hermawan  ->  jaringan komputer
  [DOSEN -> mengajar -> MATKUL]
    bapak anggasoft spoken  ->  basis data
    ibu neny kurniati  ->  pemrograman web
    bapak anggay lury  ->  kecerdasan buatan
    bapak machlul alamin  ->  jaringan komputer

  SISTEM QUERY SEMANTIK - AKAD